<a href="https://colab.research.google.com/github/nyp-sit/iti121-2026s2/blob/main/L4/02_finetune_unfreeze_last_layers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 2 — Unfreezing the Last Conv Layer (Two-Stage Fine-tuning)

In Notebook 1 the whole convolutional base was **frozen**: only the new FC layer learned. Here we also **fine-tune the last convolutional block (`layer4`)** of ResNet18 and compare with the frozen approach.

We do this with **two-stage training**:

| Stage | Frozen | Trainable | Learning rate |
|---|---|---|---|
| **Stage 1** — train the head | entire conv base | `fc` | 1e-3 (normal) |
| **Stage 2** — fine-tune | `conv1` … `layer3` | `layer4` + `fc` | 1e-4 (10× smaller) |

and compare against continuing to train with the base fully frozen.

### Main ideas in this notebook
1. **Why unfreeze the *last* layer?** Feature hierarchy: early layers = generic, late layers = task-specific.
2. **Selective unfreezing** with `requires_grad`.
3. **Two-stage training** — first train the new head, *then* unfreeze and fine-tune with a small learning rate.
4. **Trade-offs**: accuracy vs training time vs overfitting risk.

## 1. Setup, data and helpers
(Same as Notebook 1 — same fixed train/val split so results are comparable.)

In [ ]:
import os, time, copy, random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.datasets.utils import download_and_extract_archive

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed()

# Pick the fastest available device: NVIDIA GPU -> Apple Silicon GPU -> CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("PyTorch", torch.__version__, "| device:", device)

In [ ]:
DATA_URL  = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
DATA_ROOT = "data"
DATA_DIR  = os.path.join(DATA_ROOT, "flower_photos")

if not os.path.isdir(DATA_DIR):
    download_and_extract_archive(DATA_URL, download_root=DATA_ROOT)   # ~220 MB, only downloaded once

# One sub-folder per class -> ImageFolder labels the images automatically
class_names = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))
for c in class_names:
    print(f"{c:12s} {len(os.listdir(os.path.join(DATA_DIR, c)))} images")

In [ ]:
# ResNet18 was trained on ImageNet, so we must feed it images prepared the SAME way:
# 224x224 pixels, normalised with the ImageNet channel mean and standard deviation.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def make_datasets(train_tf, val_tf, val_frac=0.2, seed=SEED):
    """Same fixed 80/20 split in every notebook, so results are comparable."""
    n = len(datasets.ImageFolder(DATA_DIR))
    idx = np.random.RandomState(seed).permutation(n)
    n_val = int(val_frac * n)
    val_idx, train_idx = idx[:n_val], idx[n_val:]
    # Two ImageFolder objects over the same files so train/val can use different transforms
    train_ds = Subset(datasets.ImageFolder(DATA_DIR, transform=train_tf), train_idx)
    val_ds   = Subset(datasets.ImageFolder(DATA_DIR, transform=val_tf),   val_idx)
    return train_ds, val_ds

BATCH_SIZE  = 32
NUM_WORKERS = 2          # set to 0 if you get DataLoader errors (e.g. on Windows)
PIN = device.type == "cuda"

def make_loaders(train_ds, val_ds, shuffle_train=True):
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=shuffle_train,
                              num_workers=NUM_WORKERS, pin_memory=PIN)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=PIN)
    return train_loader, val_loader

train_ds, val_ds = make_datasets(train_tf=eval_tf, val_tf=eval_tf)   # no augmentation yet
train_loader, val_loader = make_loaders(train_ds, val_ds)
print(f"train: {len(train_ds)} | val: {len(val_ds)}")

In [ ]:
def set_train_mode(model):
    """model.train(), but keep BatchNorm layers of FROZEN blocks in eval mode.

    In train mode BatchNorm keeps updating its running mean/variance even when its
    weights are frozen - that would silently change the 'frozen' features.
    """
    model.train()
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d) and not any(p.requires_grad for p in m.parameters()):
            m.eval()

def run_epoch(model, loader, criterion, optimizer=None):
    """One pass over `loader`. Trains if an optimizer is given, otherwise just evaluates."""
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss = criterion(out, y)
        if optimizer is not None:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct    += (out.argmax(1) == y).sum().item()
        n          += x.size(0)
    return total_loss / n, correct / n

def fit(model, train_loader, val_loader, optimizer, epochs, scheduler=None, verbose=True):
    """Standard training loop. Returns the history and restores the best weights (by val acc)."""
    criterion = nn.CrossEntropyLoss()
    hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "epoch_time": []}
    best_acc, best_state = -1.0, None
    for ep in range(1, epochs + 1):
        t0 = time.time()
        set_train_mode(model)
        tl, ta = run_epoch(model, train_loader, criterion, optimizer)
        model.eval()
        with torch.no_grad():
            vl, va = run_epoch(model, val_loader, criterion)
        if scheduler is not None:
            scheduler.step()
        dt = time.time() - t0
        for k, v in zip(hist, (tl, ta, vl, va, dt)):
            hist[k].append(v)
        if va > best_acc:
            best_acc, best_state = va, copy.deepcopy(model.state_dict())
        if verbose:
            print(f"epoch {ep:2d}/{epochs} | train loss {tl:.3f} acc {ta:.3f} | "
                  f"val loss {vl:.3f} acc {va:.3f} | {dt:.1f}s")
    model.load_state_dict(best_state)
    hist["best_val_acc"] = best_acc
    return hist

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"trainable params: {trainable:,} / {total:,}  ({100*trainable/total:.2f}%)")
    return trainable

## 2. Why unfreeze the *last* layer?

CNNs learn a **hierarchy of features**:

```
conv1 / layer1  →  edges, colours, simple textures          (generic: useful for any image)
layer2          →  corners, repeated patterns
layer3          →  object parts (petals? wheels? eyes?)
layer4          →  high-level, ImageNet-specific concepts   (most task-specific)
fc              →  ImageNet's 1000 classes                  (replaced)
```

The early layers transfer almost perfectly to flowers, so we keep them frozen. The last block (`layer4`) encodes concepts tuned for ImageNet classes; letting it adapt (**fine-tune**) to flower images can improve accuracy.

Why not unfreeze everything? With only ~3k training images, updating all 11M weights risks **overfitting** and **destroying** useful pre-trained features, and it is slower.

Let's see how many parameters each block holds:

In [ ]:
m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
total = sum(p.numel() for p in m.parameters())
for name, module in m.named_children():
    n = sum(p.numel() for p in module.parameters())
    if n: print(f"{name:7s} {n:>10,} params  ({100*n/total:5.1f}%)")

`layer4` alone contains over 70 % of all weights, so unfreezing just this one block already adds a lot of capacity.

## 3. Why two-stage training?

When we replace `fc`, the new layer starts with **random weights**. Its first predictions are essentially guesses, so the loss and the gradients are large and noisy.

* If `layer4` were unfrozen **from the very first step**, those large, noisy gradients would flow back into the pre-trained `layer4` and could **damage the useful features** it already learned (sometimes called *catastrophic forgetting*).
* So we first **freeze the base and train only the head** (Stage 1 — exactly Notebook 1). Once the head makes sensible predictions, the gradients become small and meaningful.
* Then we **unfreeze `layer4`** and continue training head + `layer4` together with a **small learning rate** (Stage 2), so the pre-trained weights are only gently adjusted.

Because both stages use a single learning rate for all trainable parameters, we can use a plain optimizer — no parameter groups needed.

## 4. Helper functions

In [ ]:
def build_frozen_resnet18(num_classes):
    """Stage-1 model: pre-trained ResNet18, whole base frozen, new trainable fc."""
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    for p in model.parameters():
        p.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(device)

def unfreeze(model, block_name):
    """Make one top-level block (e.g. 'layer4') trainable again."""
    for p in getattr(model, block_name).parameters():
        p.requires_grad = True

def trainable_params(model):
    """Only parameters with requires_grad=True are given to the optimizer."""
    return [p for p in model.parameters() if p.requires_grad]

def concat_hist(h1, h2):
    """Join the histories of two training stages into one for plotting."""
    keys = ["train_loss", "train_acc", "val_loss", "val_acc", "epoch_time"]
    return {k: h1[k] + h2[k] for k in keys}

## 5. Stage 1 — train only the new head (base frozen)

This is exactly Approach A from Notebook 1.

In [ ]:
STAGE1_EPOCHS = 3
STAGE2_EPOCHS = 4
HEAD_LR       = 1e-3    # stage 1: new, randomly-initialised head
FINETUNE_LR   = 1e-4    # stage 2: 10x smaller, to only gently adjust pre-trained weights

set_seed()
model_stage1 = build_frozen_resnet18(len(class_names))
count_params(model_stage1)

opt1 = optim.Adam(trainable_params(model_stage1), lr=HEAD_LR)   # = only fc
hist_stage1 = fit(model_stage1, train_loader, val_loader, opt1, epochs=STAGE1_EPOCHS)

## 6. Stage 2 — unfreeze `layer4` and fine-tune

We start from a **copy** of the Stage-1 model (the head is already trained), make `layer4` trainable, and create a **new optimizer** with the small learning rate.

> ⚠️ Always create a new optimizer after changing `requires_grad`: the old optimizer only knows about the parameters it was given when it was created (here: just `fc`).

Note that the BatchNorm layers inside `layer4` are now trainable, so `set_train_mode` puts them back into train mode, while the BatchNorm layers in the still-frozen blocks stay in eval mode.

In [ ]:
set_seed()
model_ft = copy.deepcopy(model_stage1)      # continue from the trained head
unfreeze(model_ft, "layer4")
count_params(model_ft)

opt2 = optim.Adam(trainable_params(model_ft), lr=FINETUNE_LR)   # layer4 + fc, one learning rate
hist_stage2 = fit(model_ft, train_loader, val_loader, opt2, epochs=STAGE2_EPOCHS)
hist_ft = concat_hist(hist_stage1, hist_stage2)

## 7. Baseline — keep the base frozen for the same number of epochs

For a fair comparison, we also continue the Stage-1 model for the **same** number of extra epochs, but with the base still frozen (only `fc` keeps training). Both models then have seen the data for `STAGE1_EPOCHS + STAGE2_EPOCHS` epochs in total.

In [ ]:
set_seed()
model_frozen = copy.deepcopy(model_stage1)  # base stays frozen
count_params(model_frozen)

opt_f = optim.Adam(trainable_params(model_frozen), lr=HEAD_LR)
hist_more = fit(model_frozen, train_loader, val_loader, opt_f, epochs=STAGE2_EPOCHS)
hist_frozen = concat_hist(hist_stage1, hist_more)

## 8. Results

The dotted vertical line marks the switch from Stage 1 to Stage 2.

In [ ]:
runs = {"frozen base (fc only)": hist_frozen, "two-stage: unfreeze layer4": hist_ft}

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for (name, h), c in zip(runs.items(), ["tab:blue", "tab:orange"]):
    ep = range(1, len(h["val_acc"]) + 1)
    axes[0].plot(ep, h["train_loss"], "--", color=c, label=f"{name} (train)")
    axes[0].plot(ep, h["val_loss"],   "-",  color=c, label=f"{name} (val)")
    axes[1].plot(ep, h["val_acc"], marker="o", color=c, label=name)
    axes[2].plot(ep, np.array(h["train_acc"]) - np.array(h["val_acc"]), marker="o", color=c, label=name)
axes[0].set_title("Loss (dashed = train, solid = val)")
axes[1].set_title("Validation accuracy")
axes[2].set_title("Generalisation gap (train acc − val acc)")
for ax in axes:
    ax.axvline(STAGE1_EPOCHS + 0.5, color="gray", ls=":")
    ax.set_xlabel("epoch"); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
print(f"{'strategy':28s} {'trainable':>11s} {'best val acc':>13s} {'final train acc':>16s} {'stage-2 s/epoch':>16s}")
for (name, h), m, h2 in zip(runs.items(), [model_frozen, model_ft], [hist_more, hist_stage2]):
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"{name:28s} {trainable:>11,} {max(h['val_acc']):>13.3f} {h['train_acc'][-1]:>16.3f} "
          f"{np.mean(h2['epoch_time']):>16.1f}")

### How to read the results

* **Accuracy** — after the switch to Stage 2, the fine-tuned model usually gets a clear boost in validation accuracy, while the frozen baseline improves only slowly: the high-level features of `layer4` can now specialise to petals, colours and shapes of flowers.
* **No collapse at the switch** — because the head was trained first and we use a small learning rate, accuracy does not drop when `layer4` is unfrozen. (Try setting `FINETUNE_LR = 1e-2` to see what happens when the pre-trained weights are changed too aggressively.)
* **Overfitting** — look at the generalisation gap. With ~8.4 M trainable parameters and only ~2,900 training images, training accuracy approaches 100 % quickly → the model starts to memorise. Notebook 3 tackles this with **data augmentation**.
* **Speed** — Stage-2 epochs are slower because gradients must be back-propagated through `layer4`. The feature-caching trick from Notebook 1 also no longer applies, because `layer4`'s output changes at every step. (Stage 1, however, *can* still use cached features!)

> Results vary slightly from run to run and between CPU/GPU. Try changing `STAGE1_EPOCHS`, `STAGE2_EPOCHS` or `FINETUNE_LR`.

## 9. Summary

| | Frozen base | Two-stage: + `layer4` |
|---|---|---|
| Trainable params | ~2.5 k | ~8.4 M |
| Can cache features? | ✅ yes | Stage 1 ✅ / Stage 2 ❌ |
| Speed per epoch | fast | slower in Stage 2 |
| Adaptation to new data | low | higher |
| Overfitting risk (small data) | low | higher |

**Key takeaways**
* Unfreeze from the **top** (closest to the output) — `layer4` holds the most task-specific features.
* **Two-stage training**: train the new head with the base frozen first, then unfreeze and fine-tune with a **small learning rate** so the pre-trained features aren't destroyed by noisy gradients from a random head.
* After changing `requires_grad`, create a **new optimizer** with only the trainable parameters.
* Keep frozen BatchNorm layers in eval mode.
* More trainable layers → more capacity → more risk of overfitting. Next: **Notebook 3** combines fine-tuning with **data augmentation**.